In [ ]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
import hmac, hashlib

load_dotenv(find_dotenv(usecwd=True))
engine = create_engine(os.environ['DATABASE_URL'])

In [ ]:
colunas_origem_bruto = [
    'identifier','call_date','call_mode','speaking_with_agent_time','recording', #COLUNAS ESSENCIAIS
    'qualification_name','qualification_behavior_text','list_id', #COLUNAS DIM_CARACTERISTICAS PARA CRUZAMENTO
    'agent_name', #COLUNA DIM_AGENTE PARA CRUZAMENTO
    'campaign_name' #COLUNA DIM_FILIAL PARA CRUZAMENTO
]

In [ ]:
caminho_leitura_bruto = 'Arquivos CSV/Diario/12.08_a_12.09_ligacoes_detalhadas.csv'
df_bruto = pd.read_csv(caminho_leitura_bruto, usecols= colunas_origem_bruto)

In [ ]:
df_bruto = df_bruto.rename(columns={
    'call_date': 'data_hora',
    'call_mode': 'tipo_ligacao',
    'speaking_with_agent_time': 'tempo_falado_seg',
    'recording': 'audio_url',
    'qualification_name': 'qualificacao',
    'qualification_behavior_text': 'comportamento',
    'list_id': 'lista_id',
    'agent_name': 'nome_agente',
    'campaign_name': 'nome_campanha'
})

In [ ]:
traducao_ligacao = {
    'dialer': 'discador',
    'acw_manual': 'pós-atendimento manual'
}

df_bruto['tipo_ligacao'] = df_bruto['tipo_ligacao'].map(traducao_ligacao).fillna(df_bruto['tipo_ligacao']).fillna('Não Informado')

In [ ]:
traducao_comportamento = {
    'not-call-identifier': 'sem-identificador-de-chamada',
    'not-call-phone': 'não-ligar-telefone',
    'repeat': 'repetir'
}

df_bruto['comportamento'] = df_bruto['comportamento'].map(traducao_comportamento).fillna(df_bruto['comportamento']).fillna('Não Informado')

In [ ]:
df_bruto['tempo_falado_seg'] = pd.to_timedelta(df_bruto['tempo_falado_seg']).dt.total_seconds().fillna(0).astype(int)

In [ ]:
df_bruto['data_hora'] = pd.to_datetime(df_bruto['data_hora'], format='%d/%m/%Y %H:%M:%S')
df_bruto['id_data'] = df_bruto['data_hora'].dt.strftime('%Y%m%d').astype(int)

In [ ]:
segredo = os.environ['HASH_SECRET'].encode()

def hmac_hex(texto):
    return hmac.new(segredo, texto.encode(), hashlib.sha256).hexdigest()

df_bruto['cliente_hash'] = df_bruto['identifier'].map(
    lambda x: hmac_hex(x) if pd.notna(x) else None
)

codigo = df_bruto['audio_url'].str.extract(r'/calls/([^/?#]+)')[0]
chave_reserva = (df_bruto['id_data'].astype(str) + df_bruto['data_hora'].astype(str)
                + df_bruto['tempo_falado_seg'].astype(str) + df_bruto['identifier'].astype(str))
reserva = 'sem_audio_' + chave_reserva.map(lambda x: hmac_hex(x)[:16])

df_bruto['codigo_chamada'] = codigo.fillna(reserva)

print("sem código na audio_url:", codigo.isna().sum())
print("clientes distintos:", df_bruto['cliente_hash'].nunique())
assert df_bruto['codigo_chamada'].is_unique, "codigo_chamada repetido"

df_bruto = df_bruto.drop(columns=['identifier'])

In [ ]:
df_bruto['tipo_telefone'] = np.random.choice(['móvel','fixo'], size=len(df_bruto), p=[0.95,0.05])
df_bruto['finalizado_agente'] = np.random.choice([True,False], size=len(df_bruto))

In [ ]:
df_bruto[['sigla_uf', 'loja']] = df_bruto['nome_campanha'].str.split(' ', n=1, expand=True)

In [ ]:
df_bruto['lista_id'] = df_bruto['lista_id'].fillna(-1).astype(int)
df_bruto['qualificacao'] = df_bruto['qualificacao'].fillna('Não Informado')

In [ ]:
dim_agente_conf = pd.read_csv('Arquivos CSV/dim_agentes_confidencial.csv')
dim_filial_conf = pd.read_csv('Arquivos CSV/dim_filial_confidencial.csv')

In [ ]:
n_linhas = len(df_bruto)

df_fato = (
    df_bruto
    .merge(dim_agente_conf[['nome_agente', 'nome_oculto']], on='nome_agente', how='left')
    .rename(columns={'nome_oculto': 'agente_oculto'})
    .merge(dim_filial_conf[['sigla_uf', 'loja', 'nome_oculto']], on=['sigla_uf', 'loja'], how='left')
    .rename(columns={'nome_oculto': 'filial_oculto'})
)

assert len(df_fato) == n_linhas, "merge duplicou linhas"
assert df_fato['agente_oculto'].notna().all(), df_fato.loc[df_fato['agente_oculto'].isna(), 'nome_agente'].unique()
assert df_fato['filial_oculto'].notna().all(), df_fato.loc[df_fato['filial_oculto'].isna(), ['sigla_uf', 'loja']].drop_duplicates()

df_fato = df_fato.drop(columns=['nome_agente', 'loja', 'nome_campanha'])
df_fato = df_fato.rename(columns={'agente_oculto': 'nome_agente', 'filial_oculto': 'loja'})

In [ ]:
dim_agente_pg = pd.read_sql('SELECT id_agente, nome_agente FROM dim_agente', engine)
dim_filial_pg = pd.read_sql('SELECT id_filial, loja, sigla_uf FROM dim_filial', engine)
dim_carac_pg  = pd.read_sql('SELECT id_caracteristicas, lista_id, qualificacao, comportamento FROM dim_caracteristicas', engine)

df_final = (
    df_fato
    .merge(dim_agente_pg, on='nome_agente', how='left')
    .merge(dim_filial_pg, on=['sigla_uf', 'loja'], how='left')
    .merge(dim_carac_pg, on=['lista_id', 'qualificacao', 'comportamento'], how='left')
)

assert len(df_final) == n_linhas, "merge com o banco duplicou linhas"
for col in ['id_agente', 'id_filial', 'id_caracteristicas']:
    assert df_final[col].notna().all(), f"{col} sem correspondência no banco"

colunas_banco = [
    'id_data', 'id_agente', 'id_filial', 'id_caracteristicas',
    'codigo_chamada', 'cliente_hash', 'data_hora', 'tipo_telefone', 'tipo_ligacao',
    'tempo_falado_seg', 'finalizado_agente', 'audio_url'
]
df_final = df_final[colunas_banco].astype(
    {'id_agente': int, 'id_filial': int, 'id_caracteristicas': int}
)
df_final.head()

In [ ]:
df_final.to_sql('fato_ligacoes', engine, if_exists='append',
                index=False, method='multi', chunksize=1000)

In [ ]:
pd.read_sql('SELECT COUNT(*) AS total FROM fato_ligacoes', engine)   # esperado: 15124